In [1]:
import os
import json
import urllib.request
import importlib, pathlib, sys
from urllib.parse import urlparse
from datetime import datetime, timezone

import teehr
import pandas as pd
from teehr.evaluation.spark_session_utils import create_spark_session

from teehr import DeterministicMetrics as dm
from teehr import Signatures as s
from teehr import RowLevelCalculatedFields as rcf
from teehr import TimeseriesAwareCalculatedFields as tcf
from teehr import Bootstrappers as bs

from teehr.models.filters import TableFilter

from pyspark.sql import functions as F

from pyspark.sql import DataFrame

import copy
import time

teehr.__version__

'0.7.0'

In [2]:


BRANCH = "improve-nwmd-preprocessing-efficiency"
MOD_DIR = pathlib.Path("/home/jovyan/nwmd_modules")
MOD_DIR.mkdir(parents=True, exist_ok=True)

RAW = ("https://raw.githubusercontent.com/RTIInternational/teehr-hub/"
       f"{BRANCH}/warehouse/remote/03_preprocessing/nwm_diagnostics/utils.py")

# cache-buster: raw.githubusercontent caches branch URLs for ~5 minutes
urllib.request.urlretrieve(f"{RAW}?t={time.time()}", MOD_DIR / "utils.py")

sys.path.insert(0, str(MOD_DIR))
import utils
importlib.reload(utils)  # picks up a re-download without a kernel restart
print("loaded", utils.__file__)


loaded /home/jovyan/nwmd_modules/utils.py


In [3]:
from dataclasses import dataclass, field
from typing import Dict, List, Tuple

# ---------------------------------------------------------------------------
# Declarative dimension spec
# ---------------------------------------------------------------------------
# Every group-by column of the output table is declared once, below, and the
# calculated fields, row expansions, group_by lists, nullable_fields and
# partition_by are all *derived* from those declarations. Adding a dimension
# (e.g. water_year) is a single spec entry rather than matching edits in eight
# hand-maintained places.

# Columns of fcst_joined_timeseries that do NOT identify a timeseries.
NON_UNIQUE_FIELDS = [
    "primary_value", "secondary_value", "created_at", "updated_at", "value_time",
]
# Identifies a single forecast, so it must be a key of the bin aggregation, but
# is aggregated away before the final metrics.
BIN_ONLY_FIELDS = ["reference_time"]
# Nullable in the output regardless of the dimension specs.
ALWAYS_NULLABLE = ["member"]

# teehr builds MERGE partition filters by running SELECT DISTINCT / MIN-MAX over
# the *lazy* source view, which executes the entire bootstrap DAG once before the
# MERGE executes it again -- 2x the most expensive stage in the pipeline, to prune
# partitions we gain little from pruning. Keeping them off is also what makes a
# nullable partition column safe (see DimensionSpec.nullable_partition_fields).
USE_PARTITION_FILTERS = False

# Dimension stages. The distinction matters for both correctness and cost:
PRE_BIN = "pre_bin"    # levels select row SUBSETS -> must expand before the bin agg
BIN = "bin"            # plain grouping key, no expansion
POST_BIN = "post_bin"  # levels keep all rows -> expand after the bin agg


@dataclass(frozen=True)
class Level:
    """One level of a dimension.

    values : one SQL expression per name in Dimension.names.
    keep : boolean SQL; rows where this is false are dropped for this level.
    payload : extra output column -> source column, for dimensions that pivot
        metric *outputs* into rows (window_agg) rather than replicating rows.
    """

    values: Tuple[str, ...]
    keep: str = "true"
    payload: Dict[str, str] = field(default_factory=dict)


@dataclass(frozen=True)
class Dimension:
    """One or more output columns expanded together as correlated levels.

    names is a tuple rather than a str so a single dimension can express
    *correlated* levels -- Spark GROUPING SETS emulated by row replication,
    which teehr's flat group_by cannot express natively. Only length-1 tuples
    are needed today; see the water_year notes below for when that changes.
    """

    names: Tuple[str, ...]
    stage: str
    levels: Tuple[Level, ...] = ()
    calculated_fields: Tuple = ()       # teehr CFs that materialize the source columns
    consumes: Tuple[str, ...] = ()      # helper columns dropped after the stack
    payload_fields: Tuple[str, ...] = ()
    nullable_names: Tuple[str, ...] = ()
    partition_names: Tuple[str, ...] = ()
    in_bin_group: bool = True           # name(s) are a key of the bin aggregation
    in_final_group: bool = True         # name(s) are a key of the final aggregation

    def validate(self):
        for lvl in self.levels:
            assert len(lvl.values) == len(self.names), (
                f"{self.names}: level has {len(lvl.values)} values, "
                f"expected {len(self.names)}"
            )
            assert tuple(sorted(lvl.payload)) == tuple(sorted(self.payload_fields)), (
                f"{self.names}: every level must emit the same payload keys"
            )


@dataclass
class DimensionSpec:
    """The dimensions of one output table, plus every list derived from them."""

    entity_fields: List[str]
    dims: List[Dimension]
    # Entity columns to partition on, in addition to any dimension that sets
    # partition_names. configuration_name is always non-null, low cardinality,
    # in group_by (so it reaches the MERGE ON clause and Iceberg can prune on
    # it), and each run writes exactly one -- so a run touches one partition.
    entity_partition_fields: Tuple[str, ...] = ("configuration_name",)

    def __post_init__(self):
        for dim in self.dims:
            dim.validate()
        names = [n for d in self.dims for n in d.names]
        assert len(set(names)) == len(names), f"duplicate dimension names: {names}"
        assert not (set(names) & set(self.entity_fields)), (
            f"dimension name collides with a joined-timeseries column: "
            f"{set(names) & set(self.entity_fields)}"
        )
        assert set(self.entity_partition_fields) <= set(self.entity_fields), (
            f"unknown entity partition field(s): "
            f"{set(self.entity_partition_fields) - set(self.entity_fields)}"
        )

    def at(self, stage) -> List[Dimension]:
        return [d for d in self.dims if d.stage == stage]

    @property
    def calculated_fields(self) -> List:
        return [cf for d in self.dims for cf in d.calculated_fields]

    @property
    def group_by_bin(self) -> List[str]:
        """Keys of the per-forecast / per-lead-time-bin aggregation."""
        return list(self.entity_fields) + [
            n for d in self.dims if d.in_bin_group for n in d.names
        ]

    @property
    def group_by(self) -> List[str]:
        """Keys of the final metric aggregation (drops reference_time)."""
        return [f for f in self.entity_fields if f not in BIN_ONLY_FIELDS] + [
            n for d in self.dims if d.in_final_group for n in d.names
        ]

    @property
    def nullable_fields(self) -> List[str]:
        return list(ALWAYS_NULLABLE) + [
            n for d in self.dims for n in d.nullable_names
        ]

    @property
    def partition_by(self) -> List[str]:
        return list(self.entity_partition_fields) + [
            n for d in self.dims for n in d.partition_names
        ]

    @property
    def nullable_partition_fields(self) -> List[str]:
        """Partition columns that can be NULL.

        Iceberg itself is perfectly happy with these -- a NULL identity-partition
        value simply gets its own partition. The constraint is teehr's, and only
        when MERGE partition filters are enabled: _build_partition_filters builds
        `t.<f> IN (...)` for string columns (from SELECT DISTINCT ... WHERE <f>
        IS NOT NULL) and `t.<f> >= min AND t.<f> <= max` for numeric ones, and
        BOTH evaluate to NULL -- i.e. not matched -- for a NULL partition value.
        Those target rows would then fall outside the merge and be re-INSERTed,
        and therefore duplicated, on every upsert. Harmless while
        USE_PARTITION_FILTERS is False; asserted at the write site if it is not.
        """
        nullable = set(self.nullable_fields)
        return [f for f in self.partition_by if f in nullable]


def expand_dimension(tbl, dim: Dimension):
    """Expand one Dimension's levels into rows, materializing dim.names.

    Generates the stack() SQL from the spec, so its arity and its level list can
    never drift apart. Replaces both hand-written stack() strings.
    """
    if not dim.levels:
        return tbl  # plain grouping key, nothing to expand

    out_names = [*dim.names, "_keep_row", *dim.payload_fields]
    rows = [
        [*lvl.values, lvl.keep, *(lvl.payload[p] for p in dim.payload_fields)]
        for lvl in dim.levels
    ]
    # The stack() body may still reference columns that are not selected as base
    # columns (above_q85, quarter, mean_primary_value, ...); that is exactly how
    # the helper columns get consumed and dropped in one step.
    dropped = set(dim.consumes) | set(out_names)
    base_cols = [c for c in tbl.to_sdf().columns if c not in dropped]
    body = ",\n            ".join(", ".join(r) for r in rows)

    stacked = tbl.selectExpr(
        *base_cols,
        f"""
        stack(
            {len(rows)},
            {body}
        ) as ({", ".join(out_names)})
        """,
    )
    if any(lvl.keep != "true" for lvl in dim.levels):
        stacked = stacked.where("_keep_row")
    return stacked.select(*base_cols, *dim.names, *dim.payload_fields)


# --- The dimensions of nwmd_metrics_by_location ----------------------------

WINDOW_AGGS = (("mean", s.Average), ("min", s.Minimum), ("max", s.Maximum))
WINDOW_VALUE_FIELDS = ("primary_value", "secondary_value")
BIN_COUNT_FIELD = "n_in_bin"


def window_metrics() -> List:
    """Per-bin aggregations: one Signature metric per (window agg, value field)."""
    metrics = [
        cls(primary_field_name=fld, output_field_name=f"{label}_{fld}")
        for label, cls in WINDOW_AGGS
        for fld in WINDOW_VALUE_FIELDS
    ]
    metrics.append(
        s.Count(primary_field_name="secondary_value", output_field_name=BIN_COUNT_FIELD)
    )
    return metrics


def window_agg_dimension() -> Dimension:
    """Pivots the per-bin mean/min/max columns into rows keyed by window_agg.

    This is a third shape: it pivots metric *outputs* into rows rather than
    replicating input rows, which is what `payload` expresses. It is still a
    Dimension because window_agg genuinely is part of the output key.
    """
    return Dimension(
        names=("window_agg",),
        stage=POST_BIN,
        in_bin_group=False,  # does not exist until after the bin aggregation
        payload_fields=WINDOW_VALUE_FIELDS,
        consumes=tuple(
            f"{label}_{fld}" for label, _ in WINDOW_AGGS for fld in WINDOW_VALUE_FIELDS
        ),
        levels=tuple(
            Level(
                values=(f"'{label}'",),
                payload={fld: f"{label}_{fld}" for fld in WINDOW_VALUE_FIELDS},
            )
            for label, _ in WINDOW_AGGS
        ),
    )


# --- High-flow thresholds --------------------------------------------------
# Thresholds are CLIMATOLOGICAL: fixed percentiles of each gage's own observed
# period of record, computed once from primary_timeseries, persisted, and joined
# on at run time.
#
# They used to be derived inline by tcf.AbovePercentileEventDetection over the
# filtered joined timeseries, which was wrong three ways:
#
#   1. the percentile moved whenever the run's reference_time window moved, so
#      rows written by different runs were not comparable, and re-running a
#      single quarter silently redefined its own thresholds;
#   2. each observed hour appears in the joined table once per reference_time
#      that forecasts it, so the distribution being quantiled was weighted by
#      forecast coverage rather than being the observed distribution; and
#   3. it was grouped by configuration_name (and each configuration is a
#      separate run anyway), so `above_q85` meant something DIFFERENT for short
#      range than for medium range at the same gage -- which quietly undermined
#      comparing configurations.
#
# Reading them from primary_timeseries over the full record fixes all three, and
# replaces an applyInPandas UDF (plus its shuffle, and its habit of pulling a
# whole gage's series into pandas) with a broadcast join and a column compare.

THRESHOLD_TABLE = "nwmd_flow_thresholds"
THRESHOLD_QUANTILES = (0.85, 0.95, 0.99)


def event_col(quantile) -> str:
    """The one place a quantile maps to its event-flag column name."""
    return f"above_q{int(quantile * 100)}"


def threshold_value_col(quantile) -> str:
    """The one place a quantile maps to its joined threshold-value column."""
    return f"threshold_q{int(quantile * 100)}"


VARIABLE_JOIN_KEY = "_variable_join_key"


def variable_join_key_sql(column="variable_name") -> str:
    """SQL for the key that matches an observed variable to a forecast one.

    Mirrors teehr's own rule in JoinedTimeseriesView._perform_join: a
    variable_name is `{parameter}_{period}_{statistic}`, and for the `inst`
    statistic the PERIOD IS IGNORED. That matters here because observations
    arrive as `streamflow_none_inst` while forecasts are
    `streamflow_hourly_inst` -- the same physical quantity, and the joined
    timeseries already treats them as such. Joining thresholds on the raw
    variable_name therefore matches nothing.

    Non-inst variables must still match in full, so a daily mean cannot be
    silently compared against an instantaneous value.
    """
    # get() rather than [] : under ANSI mode an out-of-range array index raises
    # instead of returning NULL, so a variable_name with fewer than three
    # underscore-separated parts would crash the whole run. teehr's own join SQL
    # guards this with an explicit size() check.
    parts = f"split({column}, '_')"
    return (
        f"CASE WHEN get({parts}, 2) = 'inst' "
        f"THEN concat_ws('_', get({parts}, 0), get({parts}, 2)) "
        f"ELSE {column} END"
    )


def build_flow_thresholds(
    spark,
    quantiles=THRESHOLD_QUANTILES,
    output_table_name=THRESHOLD_TABLE,
    location_pattern="usgs-%",
    configuration_name=None,
):
    """Compute and persist per-gage high-flow thresholds from primary_timeseries.

    Run this once. Run it again only when you deliberately want the thresholds
    to move (say after a large observation backfill) -- every metrics run reads
    the persisted values, which is what keeps a re-run of one quarter
    comparable with its neighbours.

    Percentiles are exact (`percentile`, not `percentile_approx`) so the result
    is reproducible, and are taken over ALL observed values for a gage,
    across configuration_name, since a threshold is a property of the river
    rather than of whichever ingest produced the observation. Rows with a NULL
    value are excluded; zeros are kept, as zero flow is meaningful.

    Args:
        spark (SparkSession): The Spark session to use.
        quantiles (tuple): Percentiles to compute, as fractions.
        output_table_name (str): Table to create or replace.
        location_pattern (str): SQL LIKE pattern limiting which gages to
            compute for. None for all.
        configuration_name (str): Restrict to one observation configuration.
            None (default) uses every configuration present.
    """
    start = time.perf_counter()
    ev = teehr.RemoteReadWriteEvaluation(spark=spark, enable_spark_proxy=True)

    sdf = ev.table("primary_timeseries").to_sdf().where(F.col("value").isNotNull())
    if location_pattern:
        sdf = sdf.where(F.col("location_id").like(location_pattern))
    if configuration_name:
        sdf = sdf.where(F.col("configuration_name") == configuration_name)

    qs = list(quantiles)
    pct_list = ", ".join(str(q) for q in qs)
    grouped = sdf.groupBy("location_id", "variable_name", "unit_name").agg(
        F.expr(f"percentile(value, array({pct_list}))").alias("_pcts"),
        F.count("value").alias("n_values"),
        F.min("value_time").alias("por_start"),
        F.max("value_time").alias("por_end"),
    )

    # Long format -- one row per (location, variable, unit, quantile). Keeps the
    # table extensible to new quantiles without a schema change, and is the
    # shape a dashboard would want for "this gage's q95 is 12.4 m^3/s".
    pairs = ", ".join(f"{q}, _pcts[{i}]" for i, q in enumerate(qs))
    thresholds = (
        grouped.selectExpr(
            "location_id", "variable_name", "unit_name",
            "n_values", "por_start", "por_end",
            f"stack({len(qs)}, {pairs}) as (quantile, threshold_value)",
        )
        .withColumn("computed_at", F.current_timestamp())
    )

    # Materialize before creating the table. The exact-percentile scan over the
    # whole period of record is the slow part, and running it inside a CTAS
    # leaves the target table staged for the duration -- long enough for an
    # executor's vended S3 credentials to refresh against a table the catalog
    # cannot yet resolve, which fails the task with "Table does not exist".
    # The result is only a few rows per gage, so collecting it is cheap and
    # makes the CTAS itself near-instant.
    materialized = ev.spark.createDataFrame(
        thresholds.collect(), thresholds.schema
    )

    full_table_name = f"iceberg.teehr.{output_table_name}"
    materialized.createOrReplaceTempView("_nwmd_thresholds_src")
    ev.spark.sql(
        f"CREATE OR REPLACE TABLE {full_table_name} USING iceberg "
        f"AS SELECT * FROM _nwmd_thresholds_src"
    )
    ev.spark.sql("DROP VIEW IF EXISTS _nwmd_thresholds_src")
    ev.spark.sql(
        f"ALTER TABLE {full_table_name} SET TBLPROPERTIES ("
        f"'description' = 'Climatological high-flow thresholds per location, "
        f"from the primary_timeseries period of record')"
    )

    summary = ev.spark.sql(f"""
        SELECT count(DISTINCT location_id) AS locations,
               count(*) AS rows,
               min(n_values) AS min_obs_per_location,
               min(por_start) AS por_start,
               max(por_end) AS por_end
        FROM {full_table_name}
    """).collect()[0]
    print(
        f"Wrote {full_table_name}: {summary['rows']} rows for "
        f"{summary['locations']} locations, POR {summary['por_start']} to "
        f"{summary['por_end']}, fewest observations at any location: "
        f"{summary['min_obs_per_location']}"
    )
    print(f"{time.perf_counter() - start:.1f} s")
    return full_table_name


def load_flow_thresholds(ev, quantiles=THRESHOLD_QUANTILES,
                         table_name=THRESHOLD_TABLE):
    """Read the persisted thresholds, pivoted to one row per location.

    Returns a small Spark DataFrame keyed by (location_id, variable_name,
    unit_name) with one threshold_q* column per quantile -- small enough to
    broadcast.
    """
    full_table_name = f"iceberg.teehr.{table_name}"
    if not ev.spark.catalog.tableExists(full_table_name):
        raise RuntimeError(
            f"{full_table_name} does not exist. Run build_flow_thresholds(spark) "
            f"once before generating metrics."
        )
    sdf = ev.spark.table(full_table_name).withColumn(
        VARIABLE_JOIN_KEY, F.expr(variable_join_key_sql("variable_name"))
    )
    wide = sdf.groupBy("location_id", "unit_name", VARIABLE_JOIN_KEY).agg(*[
        F.max(
            F.when(F.col("quantile") == float(q), F.col("threshold_value"))
        ).alias(threshold_value_col(q))
        for q in quantiles
    ])

    # Report what came back. The join in join_flow_thresholds is on
    # (location, variable_name, unit_name), so a mismatch on any of those shows
    # up as silently NULL thresholds -- i.e. every row landing in the "all rows"
    # level and no events at all. Printing the keys here makes that obvious
    # instead of leaving it to be discovered in the output table. This is a
    # cheap action: the table is one row per location per quantile.
    keys = sdf.select(
        "variable_name", VARIABLE_JOIN_KEY, "unit_name"
    ).distinct().collect()
    n_locations = wide.count()
    print(
        f"Loaded thresholds for {n_locations} locations; "
        f"(variable_name -> join key, unit_name) present: "
        f"{sorted((r['variable_name'], r[VARIABLE_JOIN_KEY], r['unit_name']) for r in keys)}"
    )
    if n_locations == 0:
        raise RuntimeError(
            f"{full_table_name} produced no rows -- rebuild it with "
            f"build_flow_thresholds(spark)."
        )
    return wide


def join_flow_thresholds(tbl, thresholds, quantiles=THRESHOLD_QUANTILES):
    """Broadcast-join the per-gage threshold values onto the joined timeseries.

    Left join, so a gage with no threshold row keeps its rows: those get
    False from rcf.ThresholdValueExceeded (which coalesces a NULL comparison to
    False) and therefore appear only under the NULL "all rows" threshold level,
    rather than vanishing from the table entirely.
    """
    # Match on the PARSED variable key, not the raw variable_name -- see
    # variable_join_key_sql. Rename the right-hand keys to match the left and
    # join on NAMES, so Spark emits a single copy of each key.
    value_cols = [threshold_value_col(q) for q in quantiles]
    left_cols = tbl.to_sdf().columns  # capture before adding the join key
    right = thresholds.selectExpr(
        "location_id AS primary_location_id",
        "unit_name",
        VARIABLE_JOIN_KEY,
        *value_cols,
    )
    joined = tbl.selectExpr(
        "*", f"{variable_join_key_sql('variable_name')} AS {VARIABLE_JOIN_KEY}"
    ).join(
        F.broadcast(right),
        on=["primary_location_id", "unit_name", VARIABLE_JOIN_KEY],
        how="left",
    )
    # `select` the original columns plus the thresholds rather than dropping the
    # join key: teehr shadows .drop() on the table accessor with
    # BaseTable.drop(), which drops the TABLE FROM THE CATALOG, not columns.
    return joined.select(*left_cols, *value_cols)


def threshold_dimension(quantiles) -> Dimension:
    """High-flow levels, plus a NULL level meaning "all rows".

    PRE_BIN: each level selects a subset of rows, so the per-bin mean/min/max
    must be computed over that level's rows only.

    The flags compare the OBSERVED value against that gage's persisted
    climatological threshold. rcf.ThresholdValueExceeded is
    `coalesce(value > threshold, False)` -- strictly greater, matching the
    previous percentile detection exactly, so the flags stay comparable.

    Both the event flags and the joined threshold values are `consumes`d here,
    so they are dropped at this stack and never reach the bin aggregation as
    group keys.
    """
    events = tuple(event_col(q) for q in quantiles)
    values = tuple(threshold_value_col(q) for q in quantiles)
    return Dimension(
        names=("threshold",),
        stage=PRE_BIN,
        nullable_names=("threshold",),
        consumes=events + values,
        calculated_fields=tuple(
            rcf.ThresholdValueExceeded(
                input_field_name="primary_value",
                threshold_field_name=threshold_value_col(q),
                output_field_name=event_col(q),
            )
            for q in quantiles
        ),
        levels=(
            Level(values=("cast(null as string)",)),  # all rows, no threshold
            *(Level(values=(f"'{col}'",), keep=col) for col in events),
        ),
    )


# Temporal rollups, as grouping sets over (water_year, quarter). Selecting a
# rollup adds one level, i.e. one more copy of every post-bin row -- and the
# post-bin stream is what feeds the bootstrap, the most expensive stage in the
# pipeline. A NULL in a position means that column is aggregated ACROSS for
# that level.
#
#   quarter    -> one row per quarter          (water_year, quarter)
#   water_year -> one row per water year       (water_year, NULL)
#   all        -> one row for the whole run    (NULL,       NULL)
#
# IMPORTANT: a rollup covers the data THIS RUN read, not everything in the
# table. Running one water year at a time and upserting would overwrite the
# "all" row with just that run's data, and the per-water-year rows already in
# the table cannot be combined into it after the fact (NSE, KGE and correlation
# are not averageable, and the bin-level rows they would need are not
# persisted). So enable "all" only on a run whose reference_time filters span
# the whole period of record.
TEMPORAL_ROLLUPS = {
    "quarter":    ("cast(water_year as int)", "cast(quarter as string)"),
    "water_year": ("cast(water_year as int)", "cast(null as string)"),
    "all":        ("cast(null as int)",       "cast(null as string)"),
}
DEFAULT_ROLLUPS = ("quarter", "water_year")
# The "collapsed" value per position, used to derive which columns are nullable.
TEMPORAL_NULL_SQL = ("cast(null as int)", "cast(null as string)")


def temporal_dimension(rollups) -> Dimension:
    """water_year + quarter expanded together, as one set of correlated levels.

    They have to be one Dimension rather than two: because every quarter belongs
    to exactly one water year, an independent NULL level on water_year would
    just duplicate the per-quarter rows. A rollup is only meaningful when the
    finer column collapses with it -- which is what a grouping set expresses.
    """
    names = ("water_year", "quarter")
    unknown = [r for r in rollups if r not in TEMPORAL_ROLLUPS]
    assert not unknown, (
        f"unknown temporal rollup(s) {unknown}; choose from {list(TEMPORAL_ROLLUPS)}"
    )
    assert rollups, "at least one temporal rollup is required"
    # Canonical order, however the config happened to list them.
    values = [TEMPORAL_ROLLUPS[r] for r in TEMPORAL_ROLLUPS if r in rollups]
    nullable = tuple(
        name for i, name in enumerate(names)
        if any(v[i] == TEMPORAL_NULL_SQL[i] for v in values)
    )
    return Dimension(
        names=names,
        stage=POST_BIN,
        nullable_names=nullable,
        # Stays a partition key even when the "all" rollup makes it nullable:
        # the NULL rollup rows just land in their own Iceberg partition. See
        # DimensionSpec.nullable_partition_fields for the one caveat.
        partition_names=("water_year",),
        calculated_fields=(
            rcf.WaterYear(
                input_field_name="reference_time",  # not the "value_time" default
                output_field_name="water_year",
            ),
            rcf.GenericSQL(
                output_field_name="quarter",
                sql_statement=(
                    "CONCAT(YEAR(reference_time), '-Q', QUARTER(reference_time))"
                ),
            ),
        ),
        levels=tuple(Level(values=v) for v in values),
    )


def build_dimensions(config) -> List[Dimension]:
    """Declare every derived dimension of the output table."""
    return [
        temporal_dimension(config.get("rollups", DEFAULT_ROLLUPS)),
        Dimension(
            names=("forecast_lead_time_bin",),
            stage=BIN,
            calculated_fields=(
                rcf.ForecastLeadTimeBins(
                    bin_size=pd.Timedelta(
                        hours=config.get("forecast_lead_time_bin_hours")
                    ),
                    output_field_name="forecast_lead_time_bin",
                ),
            ),
        ),
        threshold_dimension(THRESHOLD_QUANTILES),
        window_agg_dimension(),
    ]


# --- Metrics ---------------------------------------------------------------

# Each entry yields BOTH a point estimate and its bootstrap CI, from the same
# kwargs. Previously the *_boot variants silently omitted add_epsilon, so the
# interval described a different estimator than the point value it accompanied
# and the point value could fall outside its own CI.
BOOTSTRAPPED_METRICS = (
    (dm.RelativeMean, "relative_mean", {}),
    (dm.RelativeMedian, "relative_median", {}),
    (dm.RelativeMinimum, "relative_minimum", {}),
    (dm.RelativeMaximum, "relative_maximum", {}),
    (dm.RelativeStandardDeviation, "relative_standard_deviation", {}),
    (dm.RelativeBias, "relative_bias", {"add_epsilon": True}),
    (dm.NashSutcliffeEfficiency, "nash_sutcliffe_efficiency", {"add_epsilon": True}),
    (dm.KlingGuptaEfficiency, "kling_gupta_efficiency", {"add_epsilon": True}),
    (dm.PearsonCorrelation, "pearson_correlation", {"add_epsilon": True}),
)


def build_metrics(bootstrap) -> List:
    """Final metrics: signatures, point estimates, and matching bootstrap CIs."""
    metrics = [
        s.Count(),
        s.Average(),
        s.Minimum(),
        s.Maximum(),
        # Timesteps behind each bin value, carried through the window_agg pivot.
        s.Sum(primary_field_name=BIN_COUNT_FIELD, output_field_name="n_timesteps"),
    ]
    for cls, name, kwargs in BOOTSTRAPPED_METRICS:
        metrics.append(cls(**kwargs))
        # unpack_results=True is safe as of the pinned teehr commit:
        # post_process_metric_results now derives the quantile keys statically
        # via derive_map_key_list(), so unpacking no longer triggers a
        # per-metric .first() Spark action that re-ran the entire upstream DAG
        # (the confirmed cause of the "ShuffleMapStage ... first at
        # teehr/querying/utils.py:207" crashes recorded in profiling.md).
        metrics.append(
            cls(
                output_field_name=f"{name}_boot",
                bootstrap=bootstrap,
                unpack_results=True,
                **kwargs,
            )
        )
    return metrics


def resolve_location_ids(ev, config):
    """Resolve the optional location subset for a run. None means every gage.

    Two mutually exclusive config keys, both optional:

        "location_ids": ["usgs-01010500", "usgs-01011000"]   explicit list
        "location_sample_n": 25                              N gages, spread out

    Sampling is for testing. The sample is deterministic given
    ``location_sample_seed`` (default 456) and the set of gages, ordered by a
    hash of the id rather than by the id itself: ordering by id would return
    only the lowest-numbered gages, which are geographically clustered in the
    northeast and make a poor test set.

    Note the previous inline version used ``.sample(...).limit(n)``, which is
    NOT reproducible -- ``limit`` after ``sample`` depends on partitioning, so a
    different cluster shape returned different gages and an upsert would
    accumulate a drifting partial set of locations.
    """
    explicit = config.get("location_ids")
    sample_n = config.get("location_sample_n")
    if explicit and sample_n:
        raise ValueError(
            "set only one of 'location_ids' / 'location_sample_n'"
        )

    if explicit:
        print(f"Restricting to {len(explicit)} explicitly listed location(s).")
        return list(explicit)

    if not sample_n:
        return None

    seed = config.get("location_sample_seed", 456)
    rows = (
        ev.locations.filter("id like 'usgs-%'").to_sdf()
        .select("id")
        .orderBy(F.hash(F.concat(F.col("id"), F.lit(str(seed)))))
        .limit(sample_n)
        .collect()
    )
    location_ids = sorted(r.id for r in rows)
    preview = ", ".join(location_ids[:5])
    print(
        f"SAMPLED RUN: {len(location_ids)} of all gages (seed {seed}): "
        f"{preview}{' ...' if len(location_ids) > 5 else ''}"
    )
    return location_ids


def generate_nwmd_metrics(spark, config, output_table_name):
    """Generate the teehr.nwmd_metrics_by_location table for the given config.

    config format:
        {
            "configurations": ["nwm30_medium_range"],
            "forecast_lead_time_bin_hours": 24,
            "start_reference_time": "2025-10-01T00:00",
            "end_reference_time": "2026-10-01T00:00",
            # optional; defaults to ("quarter", "water_year").
            # Add "all" for a period-of-record rollup -- see TEMPORAL_ROLLUPS
            # for what that costs and when it is valid.
            "rollups": ["quarter", "water_year"],
            "bootstrap_reps": 1000,  # optional; lower it for a smoke test
            # optional, for testing only -- omit both to run every gage.
            # Mutually exclusive; see resolve_location_ids.
            "location_sample_n": 25,
            "location_sample_seed": 456,
            "location_ids": ["usgs-01010500"]
        },

    Args:
        spark (SparkSession): The Spark session to use for processing.
        config (dict): Configuration dictionary containing necessary parameters.
        output_table_name (str): Name of the output table to store the generated metrics.
    """

    configurations = config.get("configurations")
    start_reference_time = config.get("start_reference_time")
    end_reference_time = config.get("end_reference_time")

    start = time.perf_counter()

    ev = teehr.RemoteReadWriteEvaluation(spark=spark, enable_spark_proxy=True)

    joined_cols = ev.table("fcst_joined_timeseries").to_sdf().columns
    entity_fields = [c for c in joined_cols if c not in NON_UNIQUE_FIELDS]
    print(f"Entity (uniqueness) fields: {entity_fields}")

    spec = DimensionSpec(
        entity_fields=entity_fields,
        dims=build_dimensions(config),
    )
    print(f"Dimensions: {[n for d in spec.dims for n in d.names]}")
    print(f"Final group_by: {spec.group_by}")

    filters = [
        TableFilter(
            column="configuration_name",
            operator="in",
            value=configurations
        ),
        TableFilter(
            column="reference_time",
            operator=">=",
            value=start_reference_time,
        ),
        TableFilter(
            column="reference_time",
            operator="<",
            value=end_reference_time,
        )
    ]

    location_ids = resolve_location_ids(ev, config)
    if location_ids:
        filters.append(
            TableFilter(
                column="primary_location_id",
                operator="in",
                value=location_ids
            )
        )

    # Per-gage climatological thresholds, computed once by
    # build_flow_thresholds() and joined on here so the event flags do not
    # depend on this run's time window.
    thresholds = load_flow_thresholds(ev)

    # print("number of thresholds:", thresholds.count())

    # Get raw joined timeseries with every dimension's calculated fields applied.
    tbl = (
        join_flow_thresholds(
            ev.table("fcst_joined_timeseries").filter(filters),
            thresholds,
        )
        # .add_calculated_fields(spec.calculated_fields)
    )
    # print("lngth of joined table:", tbl.count())

    tbl = tbl.add_calculated_fields(spec.calculated_fields)
    # print("lngth of joined table:", tbl.count())


    # Row-filtering dimensions expand BEFORE the bin aggregation: each level
    # selects a subset of rows, so the per-bin mean/min/max must see only that
    # level's rows.
    for dim in spec.at(PRE_BIN):
        tbl = expand_dimension(tbl, dim)

    bin_aggs_tbl = tbl.aggregate(
        group_by=spec.group_by_bin,
        metrics=window_metrics(),
    )

    # Rollup dimensions expand AFTER it. Every level keeps all rows, and each is
    # functionally determined by reference_time (itself a bin key), so the result
    # is identical to expanding pre-bin -- but without multiplying the scan, the
    # event detection and the bin-agg shuffle, which handle far more rows than
    # this post-bin stream does.
    for dim in spec.at(POST_BIN):
        bin_aggs_tbl = expand_dimension(bin_aggs_tbl, dim)

    # Configure bootstrap. reps dominates total runtime, so drop it to ~10 for a
    # cheap full-scale smoke test that still exercises the same shuffle and
    # executor-disk path, then re-run at 1000 for real numbers.
    bootstrap = bs.Stationary(
        reps=config.get("bootstrap_reps", 1000),
        seed=1234,
        quantiles=[0.025, 0.975]
    )

    group_by = spec.group_by

    results = (
        bin_aggs_tbl.aggregate(
            group_by=group_by,
            metrics=build_metrics(bootstrap),
        )
        .order_by(group_by)
        .add_geometry()
    )

    # print(results.explain(mode="simple"))

    full_table_name = f"iceberg.teehr.{output_table_name}"

    if ev.spark.catalog.tableExists(full_table_name):
        assert not (USE_PARTITION_FILTERS and spec.nullable_partition_fields), (
            f"partition filters are enabled and {spec.nullable_partition_fields} "
            f"can be NULL; those rows would go unmatched by the MERGE and be "
            f"duplicated on every upsert"
        )
        results.write_to(
            table_name=output_table_name,
            write_mode="upsert",
            # The FULL key, with nullable_fields naming the columns that need
            # null-safe (<=>) matching. _build_on_clause only applies <=> to
            # fields present in BOTH lists, so the previous
            # "group_by minus nullables" left threshold and member out of the
            # MERGE ON clause entirely -- one target row then matched every
            # threshold level, and threshold landed in the UPDATE SET clause.
            uniqueness_fields=group_by,
            nullable_fields=spec.nullable_fields,
            use_partition_filters=USE_PARTITION_FILTERS,
        )
    else:
        # Commit an EMPTY table first, then write into it. A
        # `CREATE OR REPLACE TABLE ... AS SELECT` leaves the target staged --
        # not resolvable in the REST catalog -- for the whole duration of the
        # write. teehr requests `X-Iceberg-Access-Delegation: vended-credentials`,
        # so executors fetch scoped S3 credentials from the catalog per table and
        # refresh them as they near expiry; on a multi-hour write that refresh
        # 404s with "Table does not exist" and kills the task. Short writes
        # commit before any refresh is due, which is why a smoke test passes and
        # the full run dies deep into the write.
        #
        # limit(0) prunes to an empty relation (verified: the plan becomes
        # LocalTableScan <empty>), so this costs nothing, and letting teehr do
        # the create keeps the audit columns and the partitioning.
        results.limit(0).write_to(
            table_name=output_table_name,
            write_mode="create_or_replace",
            partition_by=spec.partition_by,
        )
        # INSERT OVERWRITE into a table that now exists. Also idempotent on a
        # retry, which "append" would not be.
        results.write_to(
            table_name=output_table_name,
            write_mode="overwrite",
        )

    # Read the metric column names off the written result rather than off the metric
    # models. With unpack_results=True each bootstrap metric's MapType column is
    # replaced by one column per quantile (e.g. relative_mean_boot ->
    # relative_mean_boot_0_025, _0_975), so `metric.output_field_name` would
    # advertise columns that don't exist in the table. `.columns` is schema-only,
    # so this costs no Spark action.
    # "name" and "geometry" come from add_geometry(), not from a metric.
    non_metric_columns = set(group_by) | {"name", "geometry"}
    metric_columns = [
        c for c in results.to_sdf().columns if c not in non_metric_columns
    ]

    properties = {
        "description": "NWM diagnostics metrics by location ID",
        "group_by": ", ".join(group_by),
        "metrics": ", ".join(metric_columns)
    }

    for key, value in properties.items():
        ev.spark.sql(f"""
            ALTER TABLE {full_table_name} SET TBLPROPERTIES ('{key}' = '{value}')
        """)

    end = time.perf_counter()

    elapsed_seconds = end - start
    print(f"{elapsed_seconds:.6f} s")

    # Capture resource-usage metrics for this run BEFORE spark.stop() (the REST API
    # stops responding once the session ends). Paste the printed markdown row into
    # the Profiling table above to keep a running record.
    # n_locations = len(location_ids) if "location_ids" in globals() else "All"
    # n_days = utils._infer_days_from_filters(filters)

    # run_metrics = utils.capture_spark_run_metrics(spark, label=f"{n_locations} locations, {n_days} days")
    # utils.report_utilization(run_metrics, elapsed_seconds)

    # print(
    #     f"\n| {n_locations} | {n_days} | {bootstrap.reps} | Cluster - {utils.spark_config_summary(spark)} "
    #     f"| {elapsed_seconds:.0f}s | (see run_metrics above) |"
    # )

    # Run this against the still-live session (spark.stop() is commented out below)
    # to see the actual failure reason for the retried/failed stages from this run,
    # without needing to read it off the Spark UI by hand.
    # stage_failures = utils.get_stage_attempt_failures(spark)

    # spark.stop()

In [ ]:
pod_template_path = utils.create_ondemand_pod_template()
spark = create_spark_session(
    start_spark_cluster=True,
    # 64, not 128. `spark.jars.packages` makes the DRIVER resolve the jars and
    # serve them to every executor over its own file server -- roughly 0.7-1 GB
    # each (the two AWS SDK bundles dominate). At 128 executors that is ~100 GB
    # funnelled through one Jupyter pod: the file server saturates, the
    # spark.files.io.connectionTimeout (default 120s) fires, and executors
    # self-exit with "Unable to create executor" BEFORE running any task.
    # Confirmed on the 2026-09-09 run: exitCode=1 / reason=Error on the failed
    # pods -- not OOMKilled, not Evicted -- and executor IDs reaching 142.
    # Their shuffle output dies with them, which surfaces downstream as
    # "Missing an output location for shuffle N".
    executor_instances=64,
    executor_memory="16g",
    executor_cores=2,
    pod_template_path=pod_template_path,
    update_configs={
        "spark.sql.shuffle.partitions": 2048,
        "spark.sql.adaptive.coalescePartitions.enabled": "false",
        "spark.kubernetes.executor.annotation.cluster-autoscaler.kubernetes.io/safe-to-evict": "false",
        "spark.executorEnv.TEEHR_BOOTSTRAP_ENGINE": "vectorized",
        "spark.executor.memoryOverhead": "4g",
        # Let a slow jar fetch finish instead of self-exiting at 120s.
        "spark.files.io.connectionTimeout": "600s",
        "spark.network.timeout": "600s",
        # Ramp pods over ~2 min rather than ~15 s, so 64 executors do not all
        # pull ~1 GB of jars from the driver at once. Default is 5 pods / 1 s.
        "spark.kubernetes.allocation.batch.size": "5",
        "spark.kubernetes.allocation.batch.delay": "10s",
        # deleteOnTermination stays TRUE now. Setting it false retains every
        # dead executor pod (134 last run) and we already have the answer --
        # re-add it only if a new, unexplained executor death shows up.
    }
)

# spark = create_spark_session(
#     start_spark_cluster=True,
#     executor_instances=64,
#     executor_memory="16g",
#     executor_cores=1,
#     update_configs={
#         "spark.sql.shuffle.partitions": 1024,
#         "spark.sql.adaptive.coalescePartitions.enabled": "false",
#         "spark.kubernetes.executor.annotation.cluster-autoscaler.kubernetes.io/safe-to-evict": "false",
#         "spark.executorEnv.TEEHR_BOOTSTRAP_ENGINE": "vectorized",
#         "spark.executor.memoryOverhead": "4g",
#     }
# )

# spark = create_spark_session()

INFO:teehr.evaluation.spark_session_utils:🚀 Creating Spark session: TEEHR Evaluation
INFO:teehr.evaluation.spark_session_utils:📦 Configuring Spark cluster with container image: None
INFO:teehr.evaluation.spark_session_utils:🔍 Initial spark namespace from ENV: teehr-hub
INFO:teehr.evaluation.spark_session_utils:🔍 Connecting to Kubernetes API: https://172.20.0.1:443
INFO:teehr.evaluation.spark_session_utils:🎯 Executor namespace: teehr-hub
INFO:teehr.evaluation.spark_session_utils:🔐 Executor service account: spark (in teehr-hub)
INFO:teehr.evaluation.spark_session_utils:🔐 Using in-cluster authentication
INFO:teehr.evaluation.spark_session_utils:🔗 Setting driver host to pod IP: 10.0.2.136
INFO:teehr.evaluation.spark_session_utils:✅ Spark cluster configuration successful!
INFO:teehr.evaluation.spark_session_utils:   - Executor instances: 128
INFO:teehr.evaluation.spark_session_utils:   - Executor memory: 16g
INFO:teehr.evaluation.spark_session_utils:   - Executor cores: 2
INFO:teehr.evaluat

Wrote alternate pod template to /home/jovyan/executor-pod-template-ondemand.yaml (ephemeral-storage request: 20Gi)


INFO:teehr.evaluation.spark_session_utils:🎉 Spark session created successfully!


In [5]:
# Compute the per-gage climatological high-flow thresholds ONCE, from the
# primary_timeseries period of record, and persist them to
# iceberg.teehr.nwmd_flow_thresholds.
#
# The metrics run reads this table, so the thresholds no longer move when the
# reference_time window moves -- which is what makes a re-run of a single
# quarter comparable with its neighbours. Re-run this only when you
# deliberately want the thresholds to change, e.g. after an observation
# backfill, and expect every downstream metrics row to shift meaning when you
# do.
#
# Skip this cell if nwmd_flow_thresholds is already up to date.

# build_flow_thresholds(spark)

In [6]:
# spark.sql("USE iceberg.teehr")
# spark.sql("DROP TABLE IF EXISTS nwmd_metrics_by_location_test PURGE")
# spark.sql("DROP TABLE IF EXISTS nwmd_metrics_by_location_v2 PURGE")

In [ ]:
# "rollups" is optional and defaults to ("quarter", "water_year").
# Add "all" for a period-of-record row (water_year IS NULL AND quarter IS NULL),
# but only on a run whose reference_time range covers the whole record -- a
# rollup summarizes what THIS run read, and cannot be assembled from separate
# per-water-year runs afterwards. It also adds a third copy of every post-bin
# row, i.e. ~1.5x the bootstrap work. Drop to ["quarter"] for the cheapest run.
#
# For a cheap test run, add either of these (mutually exclusive; omit both to
# run every gage) plus "bootstrap_reps": 10 --
#     "location_sample_n": 25,        deterministic given location_sample_seed
#     "location_ids": ["usgs-01010500", ...]
configurations = [
    # {
    #     "configurations": ["nwm30_medium_range"],
    #     "forecast_lead_time_bin_hours": 24,
    #     "start_reference_time": "2023-10-01T00:00",
    #     "end_reference_time": "2026-10-01T00:00",
    #     "rollups": ["quarter", "water_year", "all"],
    # },
    {
        "configurations": ["nwm30_short_range"],
        "forecast_lead_time_bin_hours": 6,
        "start_reference_time": "2024-10-01T00:00",
        "end_reference_time": "2026-10-01T00:00",
        "rollups": ["quarter", "water_year", "all"],
        "bootstrap_reps": 10
    }
]

In [8]:
# Always release the cluster. A crash partway through used to leave the executor
# pods running -- overnight, at 22 x r5.4xlarge -- because an exception in the
# loop does not stop the SparkContext. `finally` covers a raised exception and a
# Ctrl-C/interrupt alike.
#
# Note this stops Spark on SUCCESS too, which is deliberate: there is no reason
# to keep 64 executors idle once the write has committed. The inspection cells
# below then need their own session, and `create_spark_session()` with no
# cluster is plenty for querying the result table.
#
# Caveat: `finally` cannot help if the kernel itself is killed. If that happens,
# check for orphaned pods with
#     kubectl get pods | grep exec
run_seconds = None
start = time.perf_counter()
try:
    for config in configurations:
        print(config)
        generate_nwmd_metrics(
            spark,
            config,
            output_table_name="nwmd_metrics_by_location_v2",
        )
finally:
    run_seconds = time.perf_counter() - start

    # Capture resource metrics BEFORE stopping -- the Spark REST API stops
    # answering once the session ends, and a crashed run is exactly when the
    # executor/stage numbers are worth having. Guarded so a failure here can
    # never prevent the stop below.
    try:
        run_metrics = utils.capture_spark_run_metrics(
            spark, label="nwmd full run"
        )
        utils.report_utilization(run_metrics, run_seconds)
        failures = utils.get_stage_attempt_failures(spark)
    except Exception as exc:  # noqa: BLE001
        print(f"Could not capture Spark run metrics: {exc!r}")

    try:
        spark.stop()
        print(f"Spark session stopped after {run_seconds:.0f} s.")
    except Exception as exc:  # noqa: BLE001
        print(f"spark.stop() FAILED -- check for orphaned executor pods: {exc!r}")


INFO:teehr.evaluation.evaluation:Using provided Spark session.


{'configurations': ['nwm30_short_range'], 'forecast_lead_time_bin_hours': 6, 'start_reference_time': '2024-10-01T00:00', 'end_reference_time': '2026-10-01T00:00', 'rollups': ['quarter', 'water_year', 'all']}


INFO:teehr.evaluation.evaluation:Active catalog set to iceberg.
INFO:teehr.evaluation.tables.generic_table:Getting table: fcst_joined_timeseries.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: fcst_joined_timeseries.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.fcst_joined_timeseries.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.fcst_joined_timeseries.


Entity (uniqueness) fields: ['reference_time', 'primary_location_id', 'secondary_location_id', 'configuration_name', 'unit_name', 'variable_name', 'member']
Dimensions: ['water_year', 'quarter', 'forecast_lead_time_bin', 'threshold', 'window_agg']
Final group_by: ['primary_location_id', 'secondary_location_id', 'configuration_name', 'unit_name', 'variable_name', 'member', 'water_year', 'quarter', 'forecast_lead_time_bin', 'threshold', 'window_agg']


INFO:teehr.evaluation.tables.generic_table:Getting table: fcst_joined_timeseries.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: fcst_joined_timeseries.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.fcst_joined_timeseries.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.fcst_joined_timeseries.
INFO:teehr.evaluation.dataframe_base:Setting filter [TableFilter(column='configuration_name', operator=<FilterOperators.isin: 'in'>, value=['nwm30_short_range']), TableFilter(column='reference_time', operator=<FilterOperators.gte: '>='>, value='2024-10-01T00:00'), TableFilter(column='reference_time', operator=<FilterOperators.lt: '<'>, value='2026-10-01T00:00')].


Loaded thresholds for 10269 locations; (variable_name -> join key, unit_name) present: [('streamflow_daily_mean', 'streamflow_daily_mean', 'm^3/s'), ('streamflow_none_inst', 'streamflow_inst', 'm^3/s')]


INFO:teehr.evaluation.dataframe_base:Performing the aggregation.
INFO:teehr.evaluation.dataframe_base:Performing the aggregation.
INFO:teehr.evaluation.dataframe_base:Setting order_by ['primary_location_id', 'secondary_location_id', 'configuration_name', 'unit_name', 'variable_name', 'member', 'water_year', 'quarter', 'forecast_lead_time_bin', 'threshold', 'window_agg'].
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: locations.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.locations.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.locations.
INFO:teehr.evaluation.dataframe_base:Writing to table: nwmd_metrics_by_location_v2.
INFO:teehr.evaluation.tables.generic_table:Getting table: nwmd_metrics_by_location_v2.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: nwmd_metrics_by_location_v2.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.nwmd_metrics_by_location_v2.
INFO:teehr.evaluat

{
  "label": "nwmd full run",
  "timestamp": "2026-09-09T14:19:12+00:00",
  "num_executors_seen": 128,
  "num_executors_active": 128,
  "num_executors_removed": 0,
  "total_cores_active": 256,
  "peak_executor_memory_used_gb": 0.02,
  "executor_max_memory_gb": 10.12,
  "total_gc_time_min": 7.37,
  "total_task_time_min": 14278.22,
  "total_shuffle_read_gb": 997.09,
  "total_shuffle_write_gb": 1043.97,
  "total_mem_spill_gb": 0.0,
  "total_disk_spill_gb": 0.0,
  "num_stages_completed": 54,
  "num_stages_failed": 18
}
Core utilization: 88% (14278.2 task-min / 16220.2 core-min available)
Found 18 failed stage attempt(s):

Stage 32 attempt 0 (238 complete / 22 failed tasks):
  org.apache.spark.shuffle.MetadataFetchFailedException: Missing an output location for shuffle 10 partition 34
	at org.apache.spark.MapOutputTracker$.validateStatus(MapOutputTracker.scala:1770)
	at org.apache.spark.MapOutputTracker$.$anonfun$convertMapStatuses$11(MapOutputTracker.scala:1715)
	at org.apache.spark.MapOut

Py4JJavaError: An error occurred while calling o173.sql.
: org.apache.spark.SparkException: Job aborted due to stage failure: ShuffleMapStage 32 ($anonfun$withThreadLocalCaptured$2 at CompletableFuture.java:1768) has failed the maximum allowable number of times: 4. Most recent failure reason:
org.apache.spark.shuffle.FetchFailedException
	at org.apache.spark.errors.SparkCoreErrors$.fetchFailedError(SparkCoreErrors.scala:439)
	at org.apache.spark.storage.ShuffleBlockFetcherIterator.throwFetchFailedException(ShuffleBlockFetcherIterator.scala:1253)
	at org.apache.spark.storage.ShuffleBlockFetcherIterator.next(ShuffleBlockFetcherIterator.scala:983)
	at org.apache.spark.storage.ShuffleBlockFetcherIterator.next(ShuffleBlockFetcherIterator.scala:87)
	at org.apache.spark.util.CompletionIterator.next(CompletionIterator.scala:29)
	at scala.collection.Iterator$$anon$10.nextCur(Iterator.scala:594)
	at scala.collection.Iterator$$anon$10.hasNext(Iterator.scala:608)
	at scala.collection.Iterator$$ano

In [ ]:
# The run cell stops Spark, so the inspection queries below need a session of
# their own. A plain local session is plenty for reading the result table --
# there is no reason to spin the executor cluster back up just to look at rows.
# SparkSession.getActiveSession() returns None once the session is stopped.
# Note `spark._jsc` is NOT a usable check -- it stays a live JavaObject after
# stop() and would report the session as still running.
from pyspark.sql import SparkSession

if SparkSession.getActiveSession() is None:
    spark = create_spark_session()

spark.sql("USE iceberg.teehr")

In [ ]:
spark.sql("""
SELECT *
FROM nwmd_metrics_by_location_v2 
WHERE water_year is NULL
LIMIT 10
""").show()

In [ ]:
spark.sql("""
SELECT 
    configuration_name,
    collect_set(forecast_lead_time_bin) as forecast_lead_time_bins,
    collect_set(water_year) as water_years,
    collect_set(threshold) as thresholds,
    collect_set(quarter) as quarters,
    collect_set(window_agg) as window_aggs
FROM nwmd_metrics_by_location_v2 
GROUP BY configuration_name
""").show(truncate=False)

In [ ]:
# Safety net. The run cell already stops Spark in its `finally`, so this is
# normally a no-op -- it matters when you have started an inspection session
# above, or when the run cell was never executed.
try:
    spark.stop()
    print("Spark session stopped.")
except Exception as exc:  # noqa: BLE001
    print(f"spark.stop() failed: {exc!r}")